In [1]:
import pandas as pd
df=pd.read_csv('/home/sriaparna/AI/archive/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(df.info())
print(df.describe())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


/home/sriaparna/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/sriaparna/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [2]:
df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')
print(f"Missing TotalCharges: {df['TotalCharges'].isnull().sum()}")
df.dropna(inplace=True)

Missing TotalCharges: 11


In [ ]:
import matplotlib.pyplot as plt 
import seaborn as sns
plt.figure(figsize=(6,4))
sns.countplot(x='Churn',data=df,palette='magma')
plt.title('How many customers left ?')
plt.show()
print(df['Churn'].value_counts(normalize=True)*100)

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
print(df.corr(numeric_only=True)['Churn'].sort_values(ascending=False))

In [ ]:
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df[col]=df[col].map({'Yes':1,'No':0})
df['gender']=df['gender'].map({'Female':1,'Male':0})


In [ ]:
df = pd.get_dummies(df, columns=['Contract', 'PaymentMethod', 'InternetService', 
                                 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                                 'TechSupport', 'StreamingTV', 'StreamingMovies'])
print(df.columns)

In [ ]:
if 'customerID' in df.columns:
    df.drop('customerID', axis=1, inplace=True)
df = pd.get_dummies(df)
print(df.corr()['Churn'].sort_values(ascending=False))

In [ ]:
print(df.dtypes.value_counts())

In [ ]:
print(df.describe().round(2))

In [ ]:
print(df.columns)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

X=df.drop('Churn',axis=1)
Y=df['Churn']
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

model=RandomForestClassifier(n_estimators=100,class_weight='balanced',random_state=42)

model.fit(X_train,Y_train)
predictions=model.predict(X_test)

In [ ]:
print(accuracy_score(Y_test, predictions))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
cm=confusion_matrix(Y_test,predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: Did we catch the churners?')
plt.show()

In [ ]:
print(df['Churn'].unique())

In [ ]:
probabilities = model.predict_proba(X_test)
churn_risk = probabilities[:, 1]
print(churn_risk[:5])


In [ ]:
results = pd.DataFrame({
    'Actual_Churn': Y_test,
    'Churn_Probability': churn_risk
})
print(results.sort_values(by='Churn_Probability', ascending=False).head(10))

In [ ]:
print(results.groupby('Actual_Churn')['Churn_Probability'].mean())

In [ ]:
plt.figure(figsize=(10,6))
results[results['Actual_Churn'] == 0]['Churn_Probability'].plot(
    kind='density', label='Stayed', color='blue'
)
results[results['Actual_Churn'] == 1]['Churn_Probability'].plot(
    kind='density', label='Churned', color='orange'
)
plt.title('Distribution of Churn Risk Scores')
plt.xlabel('Probability of Churn')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
import joblib
joblib.dump(model,'churn_rf_model.joblib')
primt('Successfully taken')